In [1]:
#Let's test the probabilistic programs generation!

import os
from groq import Groq
import re
import json
import sys
import torch
sys.path.insert(0, "DeGAS/src")

from optimization import optimize, compile2SOGA, compile2SOGA_text, produce_cfg, produce_cfg_text, smooth_cfg, start_SOGA, initialize_params

from PROGRAMS.likelihood import compute_likelihood

#generate some synthetic data matching the stats
import numpy as np
from param_extractor import extract_params
from mutation_prompt import build_mutation_prompt

/home/rdoz/miniconda3/envs/myenv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Key to communicate

#GROQ_MODEL = "llama-3.1-8b-instant"
#GROQ_MODEL = "llama-3.3-70b-versatile"
GROQ_MODEL = "llama-3.1-70b-versatile"
GROQ_MODEL = "openai/gpt-oss-120b"

def load_groq_api_key():
    key = os.environ.get("GROQ_API_KEY")
    if key:
        return key.strip()

    for path in (".env", ".groq_api_key"):
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8") as handle:
                for line in handle:
                    line = line.strip()
                    if line.startswith("GROQ_API_KEY="):
                        return line.split("=", 1)[1].strip().strip('"').strip("'")
                    if line and not line.startswith("#"):
                        return line

    return input("Inserisci la tua GROQ_API_KEY: ").strip()


# Initialize Groq client
GROQ_API_KEY = load_groq_api_key()
if not GROQ_API_KEY:
    raise RuntimeError("Manca la GROQ_API_KEY")

# Initialize Groq client
client = Groq(api_key=GROQ_API_KEY)

In [3]:
def groq_chat(messages):
    """Send messages to Groq API and get response."""
    chat_completion = client.chat.completions.create(
        messages=messages,
        model=GROQ_MODEL,
    )
    return chat_completion.choices[0].message.content.strip()

In [4]:
DEGAS_GRAMMAR = """
## DeGAS Language Reference

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 PROGRAM STRUCTURE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Instructions must end with ;
    instr1;
    instr2;
    instr3;

Two kinds of instruction:
    assignment    var = expr;
    conditional   if condition { 
                    program
                  } else {
                    program
                  } end if;

Variables: only the variables of the dataset are available.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 NUMBERS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Any decimal with at most 2 decimal places in [-100.00, 100.00].
    OK:    0.75    -3.14    50.00    -0.01    100.00
    NOT:   1/3     0.125    1e-2     .5

Positive numbers (weights, standard deviations): must be > 0.
    OK:    0.01    0.50    1.00    3.14    99.99

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 DISTRIBUTIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
1. Gaussian mixture
   ─────────────────────────────────────────
   gm([pi_1, ..., pi_n], [mu_1, ..., mu_n], [sigma_1, ..., sigma_n])

   Constraints:
     • All three lists must have the same length  n ≥ 1
     • pi_i > 0  and  sum(pi) = 1.00   (weights)
     • sigma_i > 0                      (standard deviations)
     • mu_i  any number in [-100, 100]  (means)

   Examples:
     gm([1.00], [0.00], [1.00])                            ← 1-component (Gaussian)
     gm([0.60, 0.40], [2.00, -1.50], [0.50, 0.80])        ← 2-component
     gm([0.30, 0.50, 0.20], [4.00, 0.00, -3.00], [0.50, 1.00, 0.50])  ← 3-component

2. Uniform
   ─────────────────────────────────────────
   uniform([start, end], 2)

   Constraints:
     • start < end
     • Both in [-100.00, 100.00]
     • The literal  2  is always the third element — do not change it

   Examples:
     uniform([0.00, 1.00], 2)
     uniform([-5.00, 5.00], 2)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 ASSIGNMENTS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    var = expr;

expr must contain AT MOST ONE multiplication (*).
When a product has a number and a variable, the NUMBER MUST COME FIRST.
No division (/) is allowed. Only +, -, *.

Legal expression forms:

    atom                        a single value
    number * var                scaled variable    (number first)
    number * distribution       scaled sample      (number first)
    var * var                   product of two variables
    atom  +  atom               sum  (neither side is a product)
    atom  -  atom               difference
    number * var + atom       one product plus one atom
    number * var - atom
    atom + number * var
    atom - number * var
    number * dist + atom      same but with a distribution
    atom + number * dist

Where  atom  =  var | number | distribution

The key rule: a single assignment line may have at most one * .
If you need two products, introduce a temporary variable:

    INVALID:  a = 2.00*b + 3.00*c;
    VALID:    a = 2.00*b;  a = a + 3.00*c;

    INVALID:  a = 2.00*b*b;
    VALID:    a = b*b; a = 2.00*a;

    INVALID:  a = b*3.00;           (number must come first)
    VALID:    a = 3.00*b;

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 CONDITIONALS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
                  if condition { 
                    program
                  } else {
                    program
                  } end if;

Condition forms:
    var == number       var != number          (equality)
    lexpr  <  number    lexpr  <= number
    lexpr  >= number    lexpr  >  number       (comparison)

    lexpr = var

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 COMPLETE EXAMPLES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 1 — Single Gaussian
a = gm([1.00], [3.50], [1.20]);

# 2 — Two-component mixture
a = gm([0.70, 0.30], [4.00, -2.00], [0.50, 1.00]);

# 3 — Mixture with linear transform (one product per line)
a = gm([0.60, 0.40], [2.00, -1.00], [0.50, 0.80]);
b = 2.00*a;
c = b + 0.50;

# 4 — Three-component heavy-tailed mixtur
a = gm([0.20, 0.60, 0.20], [0.00, 0.00, 0.00], [5.00, 1.00, 0.30]);

# 5 — Conditional latent structure
a = uniform([0.00, 1.00], 2);
if a > 0.50 {
  b = gm([1.00], [5.00], [0.80]);
} else {
  b = gm([1.00], [-2.00], [0.60]);
} end if;

# 6 — Hierarchical: scale a mixture by a latent factor
a = gm([0.50, 0.50], [1.00, -1.00], [0.50, 0.50]);
b = gm([1.00], [0.00], [0.20]);
c = a*b;

# 7 — Two products needing a temp variable (pattern to follow)
a = gm([1.00], [0.00], [1.00]);
b = 2.00*a;
b = b + 3.00*a;

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 WHAT NOT TO WRITE
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
x a = Normal(0, 1);              → a = gm([1.00], [0.00], [1.00]);
x a = gm([0.5,0.5],[0,0],[1,1]); → a = gm([0.50, 0.50], [0.00, 0.00], [1.00, 1.00]);
x a = b/c;                       → division not allowed
x a = 2.00*b + 3.00*c;          → split: a = 2.00*b :: a = a + 3.00*c;
x a = b*3.00;                   → a = 3.00*b;
x a = 2.00*b*c;                 → split: a = b*c :: a = 2.00*a;
x uniform([0, 1]);               → uniform([0.00, 1.00], 2);
x weights: [0.33, 0.33, 0.34]   → use 2 decimals summing to 1: [0.34, 0.33, 0.33]
"""

SYSTEM_PROMPT = (
    "You are an expert in probabilistic programming and Bayesian modelling.\n"
    "You write programs exclusively in DeGAS, following the reference below.\n\n"
    + DEGAS_GRAMMAR
    + """
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 RULES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Numbers: exactly 2 decimal places, range [-100.00, 100.00]
• gm weights must sum to exactly 1.00
• At most one * per assignment line
• Coefficient always before variable in products
• No division
• Only variables a, b, c, d. Do NOT put underscores or other characters in variable names.
• Reply ONLY with a valid JSON object — no prose, no markdown fences
"""
)


In [5]:
def make_init_prompt(data_stats: dict, n_programs: int = 5) -> str:
    """
    User message for initial population generation.
 
    data_stats example:
        {"n": 340, "mean": 8.4, "std": 9.1, "min": 0.1, "max": 47.3,
         "skewness": 2.1, "kurtosis": 6.4, "sample": [0.3, 1.1, 8.2]}
    """
    stats_lines = "\n".join(f"  {k}: {v}" for k, v in data_stats.items())
 
    structure_targets = [
        "unimodal      — single gm component, match the data mean and std",
        "mixture2      — two-component gm, one per apparent subpopulation",
        "mixture3      — three-component gm for skewed or multi-modal data",
        "conditional   — uniform latent + if/else branching on its value",
        "hierarchical  — product of two distributions (use temp variable)",
    ]
    targets = "\n".join(
        f"  {i+1}. {t}" for i, t in enumerate(structure_targets[:n_programs])
    )
 
    return f"""Data summary:
{stats_lines}
 
Generate exactly {n_programs} DeGAS programs that are STRUCTURALLY DIVERSE and include all the variables of the dataset.
Aim for one program per structure type below:
{targets}
 
For each program:
  1. Write a one-sentence hypothesis about the data-generating process.
  2. Choose distribution families matching the data range and shape.
  3. Write valid DeGAS code — remember at most one * per line.
 
Return a list of JSON objects, each with the following elements:
  "id"         : integer
  "hypothesis" : one sentence
  "structure"  : one-word tag (unimodal / mixture2 / mixture3 / conditional / hierarchical)
  "program"    : valid DeGAS source
 
Example shape:
{{
  [
    {{
      "id": 1,
      "hypothesis": "Data is unimodal and right-skewed.",
      "structure": "unimodal",
      "program": "a = gm([1.00], [8.00], [9.00]);"
    }},
    {{
      "id": 2,
      "hypothesis": "Data has two subpopulations, one around 5 and another around 20.",
      "structure": "mixture2",
      "program": "a = gm([0.70, 0.30], [5.00, 20.00], [2.00, 3.00]);"
    }}
  ]
}}"""

In [6]:
# ---------------------------------------------------------------------------
# 4.  Groq call wrapper
# ---------------------------------------------------------------------------
 
def call_groq(
    user_message: str,
    client,
    model: str = GROQ_MODEL,
    temperature: float = 0.9,
) -> dict:
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_message},
        ],
        temperature=temperature,
        response_format={"type": "json_object"},
    )
    raw = response.choices[0].message.content
    try:
        return json.loads(raw)
    except json.JSONDecodeError as e:
        raise ValueError(f"Non-JSON response: {e}\n---\n{raw}") from e


In [7]:
def generate_if_dataset(data_size):
    data = []
    for _ in range(data_size):
        a = np.random.normal(1, 2)
        if a < 0:
            b = a * 3 + np.random.normal(0, 1)
        else:
            b = np.random.normal(8, 1)
        data.append([a, b])
    return data

data = generate_if_dataset(1000)

In [8]:
#stats = {"var_names": ['a', 'b'], "n": 200, "mean": [4.20, 5.10], "std": [3.10, 2.80], "skewness": [1.80, -1.20],
#            "min": [0.10, 1.20], "max": [18.50, 25.30]}

#generate stats from data
stats = {
    "var_names": ['a', 'b'],
    "n": len(data),
    "mean": np.mean(data, axis=0).tolist(),
    "std": np.std(data, axis=0).tolist(),
    "skewness": (np.mean((data - np.mean(data, axis=0))**3, axis=0) / (np.std(data, axis=0)**3)).tolist(),
    "kurtosis": (np.mean((data - np.mean(data, axis=0))**4, axis=0) / (np.std(data, axis=0)**4)).tolist(),
    "min": np.min(data, axis=0).tolist(),
    "max": np.max(data, axis=0).tolist(),
}
prompt = make_init_prompt(stats, n_programs=3)
print(prompt)


Data summary:
  var_names: ['a', 'b']
  n: 1000
  mean: [1.0501454653921478, 4.30011357631357]
  std: [2.0321845837996264, 5.892962188798843]
  skewness: [-0.08460777928545797, -1.083464689023187]
  kurtosis: [2.7495935066608728, 2.839899617355012]
  min: [-4.901059926514066, -16.69118949020651]
  max: [7.923645952226427, 11.275038707583764]

Generate exactly 3 DeGAS programs that are STRUCTURALLY DIVERSE and include all the variables of the dataset.
Aim for one program per structure type below:
  1. unimodal      — single gm component, match the data mean and std
  2. mixture2      — two-component gm, one per apparent subpopulation
  3. mixture3      — three-component gm for skewed or multi-modal data

For each program:
  1. Write a one-sentence hypothesis about the data-generating process.
  2. Choose distribution families matching the data range and shape.
  3. Write valid DeGAS code — remember at most one * per line.

Return a list of JSON objects, each with the following elements:

In [9]:

# initialization
result = call_groq(make_init_prompt(stats, n_programs=5), client, temperature=0.9)
candidates = result
print(candidates)

[{'id': 1, 'hypothesis': 'Both variables arise from single Gaussian distributions matching observed means and variabilities.', 'structure': 'unimodal', 'program': 'a = gm([1.00],[1.05],[2.03]);b = gm([1.00],[4.30],[5.89]);'}, {'id': 2, 'hypothesis': 'Variable a and b each consist of two subpopulations representing low and high value clusters.', 'structure': 'mixture2', 'program': 'a = gm([0.50,0.50],[-2.00,4.00],[1.50,2.00]);b = gm([0.50,0.50],[-5.00,12.00],[3.00,4.00]);'}, {'id': 3, 'hypothesis': 'Both variables exhibit three distinct modes reflecting underlying heterogeneous processes.', 'structure': 'mixture3', 'program': 'a = gm([0.34,0.33,0.33],[-4.00,0.00,5.00],[1.00,1.50,1.00]);b = gm([0.34,0.33,0.33],[-10.00,2.00,15.00],[2.00,2.50,3.00]);'}, {'id': 4, 'hypothesis': 'A latent uniform factor decides whether the data come from a low‑mean or high‑mean Gaussian regime.', 'structure': 'conditional', 'program': 'a = uniform([0.00,1.00],2);if a > 0.50 {a = gm([1.00],[2.00],[1.00]);b = 

In [10]:
for prog in candidates:
    print(f"\nProgram ID: {prog['id']}")
    print(f"Hypothesis: {prog['hypothesis']}")
    print(f"Structure: {prog['structure']}")
    print(f"DeGAS code:\n{prog['program']}")


Program ID: 1
Hypothesis: Both variables arise from single Gaussian distributions matching observed means and variabilities.
Structure: unimodal
DeGAS code:
a = gm([1.00],[1.05],[2.03]);b = gm([1.00],[4.30],[5.89]);

Program ID: 2
Hypothesis: Variable a and b each consist of two subpopulations representing low and high value clusters.
Structure: mixture2
DeGAS code:
a = gm([0.50,0.50],[-2.00,4.00],[1.50,2.00]);b = gm([0.50,0.50],[-5.00,12.00],[3.00,4.00]);

Program ID: 3
Hypothesis: Both variables exhibit three distinct modes reflecting underlying heterogeneous processes.
Structure: mixture3
DeGAS code:
a = gm([0.34,0.33,0.33],[-4.00,0.00,5.00],[1.00,1.50,1.00]);b = gm([0.34,0.33,0.33],[-10.00,2.00,15.00],[2.00,2.50,3.00]);

Program ID: 4
Hypothesis: A latent uniform factor decides whether the data come from a low‑mean or high‑mean Gaussian regime.
Structure: conditional
DeGAS code:
a = uniform([0.00,1.00],2);if a > 0.50 {a = gm([1.00],[2.00],[1.00]);b = gm([1.00],[3.00],[2.00]);} els

In [11]:
for prog in candidates:
    #print(f"\nCompiling Program ID: {prog['program']}...")
    try:
        compiledFile = compile2SOGA_text(prog['program'])
        cfg = produce_cfg_text(compiledFile)
        smooth_cfg(cfg)
    
        # No free parameters — just evaluate the output distribution
        output_dist = start_SOGA(cfg, params_dict={})
        likelihood = compute_likelihood(output_dist, stats['var_names'], data)
        print(f"Program ID {prog['id']} likelihood: {likelihood:.4f}")

        
    except Exception as e:
        print(f"Error compiling program ID {prog['id']}: {e}")

Program ID 1 likelihood: -5.3207
Program ID 2 likelihood: -6.0337
Program ID 3 likelihood: -7.5458
Program ID 4 likelihood: -7.7006
Program ID 5 likelihood: -14.4465


In [12]:
for prog in candidates:
    print(f"\nExtracting parameters from Program ID: {prog['id']}...")
    try:
        # add to candidates the dict of parameters with their initial values and the rewritten program with parameters instead of literals
        rewritten, params_dict = extract_params(prog['program'])
        prog['params'] = params_dict
        prog['rewritten'] = rewritten
        print(f"Extracted parameters: {params_dict}")
    except Exception as e:
        print(f"Error extracting parameters from program ID {prog['id']}: {e}")


Extracting parameters from Program ID: 1...
Extracted parameters: {'mu1': 1.05, 'sigma1': 2.03, 'mu2': 4.3, 'sigma2': 5.89}

Extracting parameters from Program ID: 2...
Extracted parameters: {'mu1': -2.0, 'mu2': 4.0, 'sigma1': 1.5, 'sigma2': 2.0, 'mu3': -5.0, 'mu4': 12.0, 'sigma3': 3.0, 'sigma4': 4.0}

Extracting parameters from Program ID: 3...
Extracted parameters: {'mu1': -4.0, 'mu2': 0.0, 'mu3': 5.0, 'sigma1': 1.0, 'sigma2': 1.5, 'sigma3': 1.0, 'mu4': -10.0, 'mu5': 2.0, 'mu6': 15.0, 'sigma4': 2.0, 'sigma5': 2.5, 'sigma6': 3.0}

Extracting parameters from Program ID: 4...
Extracted parameters: {'mu1': 2.0, 'sigma1': 1.0, 'mu2': 3.0, 'sigma2': 2.0, 'mu3': -1.0, 'sigma3': 1.0, 'mu4': 6.0, 'sigma4': 3.0}

Extracting parameters from Program ID: 5...
Extracted parameters: {'mu1': 1.0, 'sigma1': 1.0, 'mu2': 2.0, 'sigma2': 1.5}


In [13]:
# Now let's do the gradient step 
#data = torch.tensor(data, dtype=torch.float64)
for prog in candidates:
    print(f"\nOptimizing Program ID: {prog['id']}...")
    try:
        compiledFile = compile2SOGA_text(prog['rewritten'])
        cfg = produce_cfg_text(compiledFile)
        smooth_cfg(cfg)
    
        # Initialize parameters
        params_dict = initialize_params(prog['params'])
        print(f"Initialized parameters: {params_dict}")
        
        # Define loss function for optimization
        loss = lambda output_dist : -compute_likelihood(output_dist, stats['var_names'], data)
        loss_list, time, number_of_iterations = optimize(cfg, params_dict, loss, n_steps=50, lr=0.01, print_progress=False)
        print(f"Program ID {prog['id']} optimization completed. Initial loss: {loss_list[0]:.4f}, Final loss: {loss_list[-1]:.4f}")
        prog['optimized_params'] = params_dict
        prog['initial_loss'] = loss_list[0]
        prog['final_loss'] = loss_list[-1]

    except Exception as e:
        print(f"Error optimizing program ID {prog['id']}: {e}")
        continue



Optimizing Program ID: 1...
Initialized parameters: {'mu1': tensor(1.0500, requires_grad=True), 'sigma1': tensor(2.0300, requires_grad=True), 'mu2': tensor(4.3000, requires_grad=True), 'sigma2': tensor(5.8900, requires_grad=True)}
Program ID 1 optimization completed. Initial loss: 5.3207, Final loss: 5.3207

Optimizing Program ID: 2...
Initialized parameters: {'mu1': tensor(-2., requires_grad=True), 'mu2': tensor(4., requires_grad=True), 'sigma1': tensor(1.5000, requires_grad=True), 'sigma2': tensor(2., requires_grad=True), 'mu3': tensor(-5., requires_grad=True), 'mu4': tensor(12., requires_grad=True), 'sigma3': tensor(3., requires_grad=True), 'sigma4': tensor(4., requires_grad=True)}
Program ID 2 optimization completed. Initial loss: 6.0337, Final loss: 5.7265

Optimizing Program ID: 3...
Initialized parameters: {'mu1': tensor(-4., requires_grad=True), 'mu2': tensor(0., requires_grad=True), 'mu3': tensor(5., requires_grad=True), 'sigma1': tensor(1., requires_grad=True), 'sigma2': ten

In [14]:
new_programs = call_groq(build_mutation_prompt(candidates, n_mutations=5, iteration=1, grammar=DEGAS_GRAMMAR), client, temperature=0.9)
print(new_programs)

/home/rdoz/PhD/LLM-guided_PP_synthesis/mutation_prompt.py:23: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  opt_val = float(opt_tensor)


[{'id': 6, 'hypothesis': 'A latent uniform factor selects between two subpopulation regimes, each with its own mixtures for a and b.', 'structure': 'conditional_mixture', 'program': 'a = uniform([0.00,1.00],2); if a > 0.50 { a = gm([0.70,0.30],[-1.00,5.00],[1.00,1.50]); b = gm([0.60,0.40],[2.00,10.00],[0.80,0.80]); } else { a = gm([0.55,0.45],[-3.00,1.00],[1.20,0.90]); b = gm([0.50,0.50],[-6.00,8.00],[1.50,1.50]); } end if;'}, {'id': 7, 'hypothesis': 'Two latent Gaussian factors combine multiplicatively to generate the observed variables.', 'structure': 'hierarchical_product', 'program': 'a = gm([0.50,0.50],[0.00,3.00],[0.80,0.80]); b = gm([0.50,0.50],[1.00,4.00],[0.70,0.70]); b = a*b;'}, {'id': 8, 'hypothesis': 'Each observed variable is modeled as a two‑component Gaussian mixture to capture multimodality.', 'structure': 'mixture2', 'program': 'a = gm([0.50,0.50],[0.50,1.60],[1.50,2.60]); b = gm([0.50,0.50],[3.50,5.10],[5.00,6.80]);'}, {'id': 9, 'hypothesis': 'A latent uniform factor 

In [15]:
for prog in new_programs:
    print(f"\nProgram ID: {prog['id']}")
    print(f"Hypothesis: {prog['hypothesis']}")
    print(f"Structure: {prog['structure']}")
    print(f"DeGAS code:\n{prog['program']}")


Program ID: 6
Hypothesis: A latent uniform factor selects between two subpopulation regimes, each with its own mixtures for a and b.
Structure: conditional_mixture
DeGAS code:
a = uniform([0.00,1.00],2); if a > 0.50 { a = gm([0.70,0.30],[-1.00,5.00],[1.00,1.50]); b = gm([0.60,0.40],[2.00,10.00],[0.80,0.80]); } else { a = gm([0.55,0.45],[-3.00,1.00],[1.20,0.90]); b = gm([0.50,0.50],[-6.00,8.00],[1.50,1.50]); } end if;

Program ID: 7
Hypothesis: Two latent Gaussian factors combine multiplicatively to generate the observed variables.
Structure: hierarchical_product
DeGAS code:
a = gm([0.50,0.50],[0.00,3.00],[0.80,0.80]); b = gm([0.50,0.50],[1.00,4.00],[0.70,0.70]); b = a*b;

Program ID: 8
Hypothesis: Each observed variable is modeled as a two‑component Gaussian mixture to capture multimodality.
Structure: mixture2
DeGAS code:
a = gm([0.50,0.50],[0.50,1.60],[1.50,2.60]); b = gm([0.50,0.50],[3.50,5.10],[5.00,6.80]);

Program ID: 9
Hypothesis: A latent uniform factor chooses between two thr

In [17]:
for prog in new_programs:
    #print(f"\nCompiling Program ID: {prog['program']}...")
    try:
        #print(prog['program'])
        compiledFile = compile2SOGA_text(prog['program'])
        cfg = produce_cfg_text(compiledFile)
        smooth_cfg(cfg)
    
        # No free parameters — just evaluate the output distribution
        output_dist = start_SOGA(cfg, params_dict={})
        likelihood = compute_likelihood(output_dist, stats['var_names'], data)
        print(f"Program ID {prog['id']} likelihood: {likelihood:.4f}")

        
    except Exception as e:
        print(f"Error compiling program ID {prog['id']}: {e}")
        continue

Program ID 6 likelihood: -6.2367
Program ID 7 likelihood: -7.8632
Program ID 8 likelihood: -5.3730
Program ID 9 likelihood: -7.2029


In [16]:
#code = 'latent = uniform([0.00, 1.00], 2); a = gm([1.00], [0.50], [1.20]); temp = 0.50 * a; b = gm([1.00], [3.00], [2.00]); if latent > 0.50 { b = temp * b; } else { b = b + 1.00; } end if; temp2 = 0.20 * latent; b = b + temp2;'
code = 'a = uniform([0.00,1.00],2); if a > 0.50 { a = gm([0.34,0.33,0.33],[-4.00,0.00,5.00],[1.00,1.50,1.00]); b = gm([0.34,0.33,0.33],[-10.00,2.00,15.00],[2.00,2.50,3.00]); } else { a = gm([0.40,0.30,0.30],[-3.50,0.50,6.00],[1.20,1.40,1.20]); b = gm([0.40,0.30,0.30],[-9.00,3.00,14.00],[2.20,2.70,3.20]); } end if;'

compiledFile = compile2SOGA_text(code)
cfg = produce_cfg_text(compiledFile)
smooth_cfg(cfg)

# No free parameters — just evaluate the output distribution
output_dist = start_SOGA(cfg, params_dict={})

In [ ]:
# Create the pipeline
n_steps = 10

# First generation
result = call_groq(make_init_prompt(stats, n_programs=5), client, temperature=0.9)
candidates = result["programs"]

# extract parameters and optimize each program
for prog in candidates:
    #print(f"\nExtracting parameters from Program ID: {prog['id']}...")
    try:
        # add to candidates the dict of parameters with their initial values and the rewritten program with parameters instead of literals
        rewritten, params_dict = extract_params(prog['program'])
        prog['params'] = params_dict
        prog['rewritten'] = rewritten
        #print(f"Extracted parameters: {params_dict}")
        compiledFile = compile2SOGA_text(prog['rewritten'])
        cfg = produce_cfg_text(compiledFile)
        smooth_cfg(cfg)
    
        # Initialize parameters
        params_dict = initialize_params(prog['params'])
        #print(f"Initialized parameters: {params_dict}")
        
        # Define loss function for optimization
        loss = lambda output_dist : -compute_likelihood(output_dist, stats['var_names'], data)
        loss_list, time, number_of_iterations = optimize(cfg, params_dict, loss, n_steps=50, lr=0.01, print_progress=False)
        print(f"Program ID {prog['id']} optimization completed. Initial loss: {loss_list[0]:.4f}, Final loss: {loss_list[-1]:.4f}")
        prog['optimized_params'] = params_dict
        prog['initial_loss'] = loss_list[0]
        prog['final_loss'] = loss_list[-1]
    except Exception as e:
        print(f"Error extracting parameters from program ID {prog['id']}: {e}")


for i in range(n_steps):
    print(f"\n--- Iteration {i+1} ---")
    new_programs = call_groq(build_mutation_prompt(candidates, n_mutations=5, iteration=i+1, grammar=DEGAS_GRAMMAR), client, temperature=0.9)
    new_candidates = new_programs['programs']
    #extract parameters and optimize each new program
    for prog in new_candidates:
        #print(f"\nExtracting parameters from Program ID: {prog['id']}...")
        try:
            # add to candidates the dict of parameters with their initial values and the rewritten program with parameters instead of literals
            rewritten, params_dict = extract_params(prog['program'])
            prog['params'] = params_dict
            prog['rewritten'] = rewritten
            #print(f"Extracted parameters: {params_dict}")
            compiledFile = compile2SOGA_text(prog['rewritten'])
            cfg = produce_cfg_text(compiledFile)
            smooth_cfg(cfg)
        
            # Initialize parameters
            params_dict = initialize_params(prog['params'])
            #print(f"Initialized parameters: {params_dict}")
            
            # Define loss function for optimization
            loss = lambda output_dist : -compute_likelihood(output_dist, stats['var_names'], data)
            loss_list, time, number_of_iterations = optimize(cfg, params_dict, loss, n_steps=50, lr=0.01, print_progress=False)
            print(f"Program ID {prog['id']} optimization completed. Initial loss: {loss_list[0]:.4f}, Final loss: {loss_list[-1]:.4f}")
            prog['optimized_params'] = params_dict
            prog['initial_loss'] = loss_list[0]
            prog['final_loss'] = loss_list[-1]
        except Exception as e:
            print(f"Error in program ID {prog['id']}: {e}")
            prog['errors'] = str(e)
            prog['final_loss'] = float('inf')


    #choose the best 5 programs between new_candidates and candidates based on final_loss and keep them for the next iteration
    all_candidates = candidates + new_candidates
    all_candidates = [prog for prog in all_candidates if 'final_loss' in prog]
    all_candidates.sort(key=lambda x: x['final_loss'])
    candidates = all_candidates[:5]
    print(f"Selected candidates for next iteration: {[prog['id'] for prog in candidates]}")

print("\n--- Final Selected Programs ---")
for prog in candidates:
    print(f"Program ID: {prog['id']}, Final Loss: {prog['final_loss']:.4f}")
    print(f"Hypothesis: {prog['hypothesis']}")
    print(f"Structure: {prog['structure']}")
    print(f"DeGAS code:\n{prog['program']}\n")  

Error extracting parameters from program ID 1: element 0 of tensors does not require grad and does not have a grad_fn
Program ID 2 optimization completed. Initial loss: 6.2717, Final loss: 6.1453
Error extracting parameters from program ID 3: element 0 of tensors does not require grad and does not have a grad_fn
Error extracting parameters from program ID 4: Expected parameter scale (Tensor of shape (1,)) of distribution Normal(loc: tensor([2.]), scale: tensor([0.])) to satisfy the constraint GreaterThan(lower_bound=0.0), but found invalid values:
tensor([0.])
Error extracting parameters from program ID 5: Expected parameter loc (Tensor of shape (2,)) of distribution MultivariateNormal(loc: torch.Size([2]), covariance_matrix: torch.Size([2, 2])) to satisfy the constraint IndependentConstraint(Real(), 1), but found invalid values:
tensor([nan, nan], grad_fn=<ExpandBackward0>)

--- Iteration 1 ---


line 1:27 no viable alternative at input 'uniform([0.00,1.00,2])'
line 1:9 token recognition error at: '('
line 1:15 no viable alternative at input 'uniform[0.00,'


KeyboardInterrupt: 